In [1]:
import numpy as np

from src.load_seqs_by_label import load_seqs_by_label
from pathlib import Path

processed_train_dir = Path("data/processed_train/")
seqs_by_label = load_seqs_by_label(processed_train_dir)

In [2]:
from src.gesture import Gesture
from src.HMM.HMM import HMM

seq = seqs_by_label[int(Gesture.WAVE)]

wave_HMM = HMM(N=10, M=100, init="uniform")

alpha_hat, c = wave_HMM.construct_forward(seq[0])
beta_hat, c = wave_HMM.construct_backward(seq[0], c)

In [3]:
print("sum alpha_hat[0] = ", alpha_hat[0])
assert(np.abs(1 - alpha_hat[0].sum()) < 1e-2)
print("sum beta_hat[-1] = ", beta_hat[-2])


sum alpha_hat[0] =  [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
sum beta_hat[-1] =  [100. 100. 100. 100. 100. 100. 100. 100. 100. 100.]


In [4]:
gamma = wave_HMM.calculate_gamma(alpha_hat, beta_hat)
xi = wave_HMM.calculate_xi(seq[0], alpha_hat, beta_hat)

# gamma rows sum to 1
assert np.allclose(gamma.sum(axis=1), 1.0)

# xi per-t sums to 1
assert np.allclose(xi.sum(axis=(1,2)), 1.0)

# consistency identities
# assert np.allclose(xi.sum(axis=2), gamma[:-1, :], atol=1e-6)  # sum over j
# assert np.allclose(xi.sum(axis=1), gamma[1:, :], atol=1e-6)   # sum over i

print(xi.sum(axis=2))

[[0.1 0.1 0.1 ... 0.1 0.1 0.1]
 [0.1 0.1 0.1 ... 0.1 0.1 0.1]
 [0.1 0.1 0.1 ... 0.1 0.1 0.1]
 ...
 [0.1 0.1 0.1 ... 0.1 0.1 0.1]
 [0.1 0.1 0.1 ... 0.1 0.1 0.1]
 [0.1 0.1 0.1 ... 0.1 0.1 0.1]]


In [5]:
alpha_hat, c = wave_HMM.construct_forward(seq[0])
beta_hat, _ = wave_HMM.construct_backward(seq[0], c)
gamma = wave_HMM.calculate_gamma(alpha_hat, beta_hat)
xi = wave_HMM.calculate_xi(seq[0], alpha_hat, beta_hat)

print(np.max(np.abs(xi.sum(axis=2) - gamma[:-1])))
print(np.max(np.abs(xi.sum(axis=1) - gamma[1:])))

print(gamma.shape)

4.163336342344337e-17
4.163336342344337e-17
(2220, 10)


In [6]:
wave_HMM.B

array([[0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01],
       [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
        0.01, 0.01, 0.0

In [7]:
wave_HMM.update_params(seq[0], gamma, xi)
assert np.allclose(wave_HMM.pi.sum(), 1.0)
assert np.allclose(wave_HMM.A.sum(axis=1), 1.0)
assert np.allclose(wave_HMM.B.sum(axis=1), 1.0)

In [8]:
wave_HMM.B

array([[0.00585586, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.07342342, 0.        ,
        0.00225225, 0.01756757, 0.04864865, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.00315315, 0.02702703, 0.        , 0.        , 0.        ,
        0.04954955, 0.        , 0.03603604, 0.        , 0.0472973 ,
        0.        , 0.        , 0.02432432, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.01756757, 0.        ,
        0.        , 0.        , 0.        , 0.05675676, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.0536036 ,
        0.        , 0.        , 0.04144144, 0.33693694, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.00135135, 0.00135135,
        0.        , 0.        , 0.04414414, 0.  

In [9]:
from src.HMM.HMM import HMM
import numpy as np

from src.load_seqs_by_label import load_seqs_by_label
from pathlib import Path

processed_train_dir = Path("data/processed_train/")
seqs_by_label = load_seqs_by_label(processed_train_dir)

seqs = seqs_by_label[int(Gesture.WAVE)] # focus on WAVE sequences

wave_HMM = HMM(N=10, M=100, init="uniform")

In [10]:
for it in range(10):
    ll = wave_HMM.fit_once(seqs)
    print(it, ll)

0 -365374.202556296


KeyboardInterrupt: 

In [11]:
hmm = HMM(N=10, M=100, init="random")  # random is usually much better than uniform
print("LL start:", hmm.score(seqs))

for it in range(10):
    hmm.fit_once(seqs)                 # updates params
    ll = hmm.score(seqs)               # LL under UPDATED params
    print(it, ll)

LL start: -365650.4662457383
0 -351904.5775909217
1 -358665.1418871687
2 -365925.3174144674
3 -371428.06644053856
4 -376774.0168808169
5 -386686.4644653168
6 -406807.5796076142
7 -445802.07420797687
8 -505124.58022426604
9 -566382.8885887874


In [4]:
import numpy as np
NUM_STATES=100
A = 0.5*np.eye(NUM_STATES)
for i in range(NUM_STATES):
    if i == NUM_STATES - 1:
        A[i, 0] = 0.5
    else:
        A[i, i+1] = 0.5

In [5]:
A

array([[0.5, 0.5, 0. , ..., 0. , 0. , 0. ],
       [0. , 0.5, 0.5, ..., 0. , 0. , 0. ],
       [0. , 0. , 0.5, ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 0.5, 0.5, 0. ],
       [0. , 0. , 0. , ..., 0. , 0.5, 0.5],
       [0.5, 0. , 0. , ..., 0. , 0. , 0.5]], shape=(100, 100))